# NASA Intelligence Chat — Evaluation & Test Suite

This notebook lets reviewers:
1. **Run the full pytest test suite** — no terminal needed
2. **Inspect test coverage** by module
3. **Run batch RAGAS evaluation** over 18 questions
4. **Visualise evaluation results**

---

## 1. Setup

In [8]:
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/suleimanadebowaleojo/Claude/Projects/NASA Intelligence Chat System


## 2. Run the test suite

All tests are offline — no API key required.

In [9]:
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short', '-q'],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0 and result.stderr:
    print('STDERR:')
    print(result.stderr[-2000:])

........................................................................ [ 75%]
........................                                                 [100%]
96 passed in 5.41s



## 3. Test coverage report

In [10]:
cov_result = subprocess.run(
    [
        sys.executable, '-m', 'pytest', 'tests/',
        '--cov=nasa_rag',
        '--cov-report=term-missing',
        '-q',
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(cov_result.stdout)
if cov_result.returncode != 0:
    print(cov_result.stderr[-1000:])

........................................................................ [ 75%]
........................                                                 [100%]

---------- coverage: platform darwin, python 3.13.5-final-0 ----------
Name                         Stmts   Miss  Cover   Missing
----------------------------------------------------------
src/nasa_rag/__init__.py         4      0   100%
src/nasa_rag/config.py          38      1    97%   177
src/nasa_rag/embedding.py      238    135    43%   268-271, 275-286, 290-293, 317-357, 418-468, 476, 485-508, 516-530, 535-575, 579
src/nasa_rag/evaluation.py      75     44    41%   56, 88-169, 256-258
src/nasa_rag/generation.py      29      0   100%
src/nasa_rag/retrieval.py       80      3    96%   72-73, 175
----------------------------------------------------------
TOTAL                          464    183    61%

FAIL Required test coverage of 70.0% not reached. Total coverage: 60.56%




## 4. Evaluation dataset overview

In [11]:
dataset_path = PROJECT_ROOT / 'data' / 'evaluation_dataset.txt'

questions = []
current_category = 'General'
categories = {}

with open(dataset_path) as f:
    for line in f:
        line = line.strip()
        if line.startswith('#') and '──' in line:
            current_category = line.strip('# ─').strip()
        elif line and not line.startswith('#'):
            questions.append({'question': line, 'category': current_category})
            categories[current_category] = categories.get(current_category, 0) + 1

print(f'Total questions: {len(questions)}\n')
print('Questions by category:')
for cat, count in categories.items():
    print(f'  {cat:<35}: {count}')
print()
print('Sample questions:')
for q in questions[:5]:
    print(f'  [{q["category"][:20]}] {q["question"]}')

Total questions: 18

Questions by category:
  Mission Overview                   : 3
  Crew                               : 3
  Emergency and Problem Analysis     : 3
  Disaster Analysis                  : 3
  Technical Details                  : 3
  Timeline and Events                : 3

Sample questions:
  [Mission Overview] What was the primary objective of the Apollo 11 mission?
  [Mission Overview] What were the main goals of the Apollo 13 mission?
  [Mission Overview] What was the purpose of the Challenger STS-51L mission?
  [Crew] Who were the crew members of the Apollo 11 mission?
  [Crew] Who were the crew members of the Apollo 13 mission?


## 5. Load existing evaluation results (if available)

In [12]:
import pandas as pd

results_path = PROJECT_ROOT / 'evaluation_results.csv'

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f'Results loaded: {len(df)} rows')
    display(df.head(10))
else:
    print('evaluation_results.csv not found.')
    print('Run: python scripts/batch_evaluate.py')
    df = None

evaluation_results.csv not found.
Run: python scripts/batch_evaluate.py


## 6. Visualise RAGAS scores

In [13]:
import matplotlib.pyplot as plt
import numpy as np

if df is not None:
    # Drop the aggregate row for per-question chart
    plot_df = df[~df['question'].str.startswith('AGGREGATE')].copy()
    plot_df = plot_df.dropna(subset=['response_relevancy', 'faithfulness'])

    if not plot_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('RAGAS Evaluation — NASA Intelligence Chat', fontsize=14)

        # Bar chart: per-question scores
        x = np.arange(len(plot_df))
        axes[0].bar(x - 0.2, plot_df['response_relevancy'], 0.4, label='Relevancy', color='steelblue')
        axes[0].bar(x + 0.2, plot_df['faithfulness'], 0.4, label='Faithfulness', color='darkorange')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels([f'Q{i+1}' for i in range(len(plot_df))], fontsize=8)
        axes[0].set_ylim(0, 1.05)
        axes[0].set_ylabel('Score')
        axes[0].set_title('Per-question scores')
        axes[0].legend()
        axes[0].axhline(0.7, color='green', linestyle='--', linewidth=0.8, alpha=0.6)

        # Box plot: distribution
        axes[1].boxplot(
            [plot_df['response_relevancy'].dropna(), plot_df['faithfulness'].dropna()],
            labels=['Response\nRelevancy', 'Faithfulness'],
        )
        axes[1].set_ylim(0, 1.05)
        axes[1].set_ylabel('Score')
        axes[1].set_title('Score distribution')
        axes[1].axhline(0.7, color='green', linestyle='--', linewidth=0.8, alpha=0.6, label='0.7 target')
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / 'docs' / 'evaluation_results.png', dpi=150)
        plt.show()

        # Summary stats
        print('\nAggregate statistics:')
        print(plot_df[['response_relevancy', 'faithfulness']].describe().round(3))
    else:
        print('No numeric rows to plot.')
else:
    print('No results to visualise.')

No results to visualise.


## 7. Run batch evaluation (requires API key + ChromaDB)

In [14]:
from nasa_rag.config import get_settings
cfg = get_settings()

if not cfg.is_configured():
    print('⚠️  OPENAI_API_KEY not set — batch evaluation skipped.')
    print('   Set it in .env and re-run this cell.')
else:
    run = input('Run batch evaluation? This will use API credits. (yes/no): ')
    if run.strip().lower() == 'yes':
        eval_result = subprocess.run(
            [sys.executable, 'scripts/batch_evaluate.py'],
            cwd=str(PROJECT_ROOT),
            capture_output=True,
            text=True,
        )
        print(eval_result.stdout[-3000:])
        if eval_result.returncode != 0:
            print('STDERR:', eval_result.stderr[-1000:])
    else:
        print('Skipped.')

Skipped.
